# Hourly → Daily Discharge Aggregation & Five-Panel Diagnostic

**Purpose:** Converts hourly observed and simulated discharge (m³/s) to
daily volumes (m³) and then builds a five-panel figure combining the double
mass curve, daily flow deviation, cumulative precipitation, reservoir water
level, and a bucket-model estimate of reservoir storage.

**What it does:**
- Reads hourly Q data from Excel, multiplies by 3600 s to get m³/h
- Groups to daily totals
- Assembles a five-panel stacked figure (panels a–e) on a shared x-axis

**Input:** `Hourly to daily.xlsx`, simulation output Excel  
**Output:** Five-panel diagnostic PNG

---

In [ ]:
import pandas as pd

df = pd.read_excel(r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\Hourly to daily.xlsx")  # or read_csv

# Parse datetime
df['DateTime'] = pd.to_datetime(df['Zeit'], format="%d.%m.%Y %H:%M", errors='coerce')

# Convert hourly m³/s → m³ per hour (multiply by 3600 seconds)
df['Obs_m3']  = df['Obs'] * 3600
df['Sim_m3']  = df['Sim'] * 3600

# Group by date and sum → daily volume in m³
df['Date'] = df['DateTime'].dt.date

daily = df.groupby('Date').agg(
    Obs_Daily_m3 = ('Obs_m3', 'sum'),
    Sim_Daily_m3 = ('Sim_m3', 'sum')
).reset_index()

print(daily.head())
daily.to_excel("daily_volume_m3.xlsx", index=False)

In [ ]:
"""
Hydrological Analysis Script  —  v5 (final)
=============================================
5 panels stacked vertically, shared x-axis (b–e):
  (a) Double Mass Curve
  (b) Daily Flow Deviation (Sim − Obs)
  (c) Cumulative Precipitation
  (d) Reservoir Water Level  [visual only]
  (e) Artificial Reservoir Storage (bucket model, m³)

Both Obs and Sim are read from the same Excel file (sim_path),
both are hourly and aggregated to daily mean before use.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec

# ─────────────────────────────────────────────────────────────────────────────
# FILE PATHS
# ─────────────────────────────────────────────────────────────────────────────
sim_path    = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Simulation_ZR\Calibration.xlsx"
precip_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
res_path    = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\NT_Water_Level.xlsx"

# Set to known total catchment reservoir capacity in m³, e.g. 5_200_000
# Leave as None to skip the comparison line in panel (e)
CATCHMENT_RESERVOIR_CAPACITY_M3 = None

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
START           = pd.Timestamp("2011-11-01")
END             = pd.Timestamp("2022-10-31")
SECONDS_PER_DAY = 86_400
VOLLSTAU        = -0.595   # full supply level of observed reservoir (m)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: build datetime for reservoir file
# ─────────────────────────────────────────────────────────────────────────────
def build_reservoir_datetime(df: pd.DataFrame) -> pd.Series:
    df.columns = [c.strip() for c in df.columns]
    if "datetime" in df.columns:
        return pd.to_datetime(df["datetime"], dayfirst=True, errors="coerce")
    if "Datum" in df.columns and "Zeit" in df.columns:
        return pd.to_datetime(
            df["Datum"].astype(str).str.strip() + " " +
            df["Zeit"].astype(str).str.strip(),
            dayfirst=True, errors="coerce"
        )
    if "Datum" in df.columns:
        return pd.to_datetime(df["Datum"], dayfirst=True, errors="coerce")
    raise ValueError("Reservoir file must contain 'datetime' or 'Datum'.")


# ─────────────────────────────────────────────────────────────────────────────
# 1. READ HOURLY DATA  (Obs + Sim from same file)
# ─────────────────────────────────────────────────────────────────────────────
df_hourly = pd.read_excel(sim_path)
df_hourly.columns = df_hourly.columns.str.strip()

if "Zeit" not in df_hourly.columns:
    raise ValueError("File must have column 'Zeit'.")
if "Obs" not in df_hourly.columns:
    raise ValueError(
        f"File must have column 'Obs'. "
        f"Columns found: {list(df_hourly.columns)}"
    )
if "Sim" not in df_hourly.columns:
    raise ValueError(
        f"File must have column 'Sim'. "
        f"Columns found: {list(df_hourly.columns)}"
    )

df_hourly["Zeit"] = pd.to_datetime(df_hourly["Zeit"], dayfirst=True, errors="coerce")
df_hourly = df_hourly.dropna(subset=["Zeit", "Obs", "Sim"])


# ─────────────────────────────────────────────────────────────────────────────
# 2. HOURLY → DAILY MEAN  (both Obs and Sim together)
# ─────────────────────────────────────────────────────────────────────────────
df_daily = (
    df_hourly.set_index("Zeit")[["Obs", "Sim"]]
             .resample("D")
             .mean()
             .reset_index()
             .rename(columns={"Obs": "Obs_Daily_m3s",
                               "Sim": "Sim_Daily_m3s"})
             .query("@START <= Zeit <= @END")
             .reset_index(drop=True)
)

# Sanity check
expected_days = (END - START).days + 1
if len(df_daily) < 0.95 * expected_days:
    print(f"[WARNING] Only {len(df_daily)} days vs {expected_days} expected. "
          "Check for gaps in the input file.")


# ─────────────────────────────────────────────────────────────────────────────
# 3. DERIVED COLUMNS
# ─────────────────────────────────────────────────────────────────────────────
df = df_daily.copy()

# Daily volumes (m³)
df["Obs_Daily_m3"] = df["Obs_Daily_m3s"] * SECONDS_PER_DAY
df["Sim_Daily_m3"] = df["Sim_Daily_m3s"] * SECONDS_PER_DAY

# Cumulative volumes for double mass curve
df["Cum_Obs_m3"]   = df["Obs_Daily_m3"].cumsum()
df["Cum_Sim_m3"]   = df["Sim_Daily_m3"].cumsum()

# Daily deviation
df["Flow_Dev_m3s"] = df["Sim_Daily_m3s"] - df["Obs_Daily_m3s"]   # m³/s (for bar plot)
df["Flow_Dev_m3"]  = df["Sim_Daily_m3"]  - df["Obs_Daily_m3"]    # m³   (for bucket)


# ─────────────────────────────────────────────────────────────────────────────
# 3b. PERFORMANCE METRICS
# ─────────────────────────────────────────────────────────────────────────────
valid    = df.dropna(subset=["Obs_Daily_m3s", "Sim_Daily_m3s"])
obs_arr  = valid["Obs_Daily_m3s"].values
sim_arr  = valid["Sim_Daily_m3s"].values

slope, _ = np.polyfit(valid["Cum_Obs_m3"], valid["Cum_Sim_m3"], 1)
pbias    = 100 * (sim_arr.sum() - obs_arr.sum()) / obs_arr.sum()
r2       = np.corrcoef(obs_arr, sim_arr)[0, 1] ** 2

print("\n── Performance Metrics ───────────────────────────────────────")
print(f"  Slope (double-mass) = {slope:.4f}")
print(f"  PBIAS               = {pbias:.2f} %")
print(f"  R²                  = {r2:.3f}")
print("──────────────────────────────────────────────────────────────\n")


# ─────────────────────────────────────────────────────────────────────────────
# 4. BUCKET MODEL — artificial reservoir
#
#   Overestimation day  (Sim > Obs, dev > 0):
#       store the excess → inflow = dev
#
#   Underestimation day (Sim < Obs, dev < 0):
#       release from storage → release = min(|dev|, current storage)
#       unmet = |dev| − release  (what reservoir couldn't cover)
#
#   Floor: storage never goes below zero
#   Peak storage across all days = minimum reservoir capacity needed
# ─────────────────────────────────────────────────────────────────────────────
n       = len(df)
storage = np.zeros(n)
inflow  = np.zeros(n)
release = np.zeros(n)
unmet   = np.zeros(n)

for i, dev in enumerate(df["Flow_Dev_m3"]):
    prev = storage[i - 1] if i > 0 else 0.0
    if dev >= 0:                        # overestimation → store
        inflow[i]  = dev
        storage[i] = prev + dev
    else:                               # underestimation → release
        demand     = abs(dev)
        rel        = min(demand, prev)
        release[i] = rel
        unmet[i]   = demand - rel
        storage[i] = prev - rel         # floor guaranteed: rel <= prev

df["ARR_Inflow_m3"]  = inflow
df["ARR_Release_m3"] = release
df["ARR_Storage_m3"] = storage
df["ARR_Unmet_m3"]   = unmet

# Summary
peak_storage = storage.max()
peak_date    = df.loc[df["ARR_Storage_m3"].idxmax(), "Zeit"]
total_unmet  = unmet.sum()
under_vol    = abs(df.loc[df["Flow_Dev_m3"] < 0, "Flow_Dev_m3"].sum())
pct_unmet    = 100 * total_unmet / under_vol if under_vol > 0 else 0.0

print("\n── Artificial Reservoir Summary ──────────────────────────────")
print(f"  Peak storage (min. capacity needed) : {peak_storage:,.0f} m³")
print(f"  Date of peak storage                : {peak_date.date()}")
print(f"  Total water stored                  : {inflow.sum():,.0f} m³")
print(f"  Total water released                : {release.sum():,.0f} m³")
print(f"  Unmet demand                        : {total_unmet:,.0f} m³  "
      f"({pct_unmet:.1f} % of underestimation volume)")
if CATCHMENT_RESERVOIR_CAPACITY_M3 is not None:
    ratio   = peak_storage / CATCHMENT_RESERVOIR_CAPACITY_M3 * 100
    verdict = "WITHIN" if peak_storage <= CATCHMENT_RESERVOIR_CAPACITY_M3 else "EXCEEDS"
    print(f"  Known catchment capacity            : "
          f"{CATCHMENT_RESERVOIR_CAPACITY_M3:,.0f} m³")
    print(f"  Artificial reservoir is {ratio:.1f} % of known capacity  →  {verdict}")
print("──────────────────────────────────────────────────────────────\n")


# ─────────────────────────────────────────────────────────────────────────────
# 5. PRECIPITATION  (hourly → daily sum)
# ─────────────────────────────────────────────────────────────────────────────
df_precip = pd.read_csv(precip_path, sep=';')
df_precip.columns = df_precip.columns.str.strip()

if "datetime" not in df_precip.columns or "precip" not in df_precip.columns:
    raise ValueError("Precip file must have columns 'datetime' and 'precip'.")
df_precip["datetime"] = pd.to_datetime(df_precip["datetime"], dayfirst=True, errors="coerce")
df_precip = df_precip.dropna(subset=["datetime"])

df_precip_daily = (
    df_precip.set_index("datetime")
             .resample("D")["precip"].sum()
             .reset_index()
             .query("@START <= datetime <= @END")
             .reset_index(drop=True)
)
df_precip_daily["precip"]     = df_precip_daily["precip"].clip(lower=0)
df_precip_daily["precip_cum"] = df_precip_daily["precip"].cumsum()


# ─────────────────────────────────────────────────────────────────────────────
# 6. RESERVOIR WATER LEVEL  (daily mean — visual only)
# ─────────────────────────────────────────────────────────────────────────────
df_res = pd.read_excel(res_path)
df_res.columns = df_res.columns.str.strip()
df_res["datetime"] = build_reservoir_datetime(df_res)

if "WL" not in df_res.columns:
    raise ValueError("Reservoir file must contain 'WL' column.")
df_res["WL"] = pd.to_numeric(
    df_res["WL"].astype(str).str.replace(",", ".", regex=False).str.strip(),
    errors="coerce"
)
df_res = df_res.dropna(subset=["datetime", "WL"]).sort_values("datetime")

df_res_daily = (
    df_res.set_index("datetime")
          .resample("D")["WL"].mean()
          .reset_index()
          .rename(columns={"datetime": "Zeit"})
          .query("@START <= Zeit <= @END")
          .reset_index(drop=True)
)


# ─────────────────────────────────────────────────────────────────────────────
# 7. PLOT
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":     "monospace",
    "font.size":       8,
    "axes.titlesize":  9,
    "axes.labelsize":  8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
})

fig = plt.figure(figsize=(7, 14))
gs  = gridspec.GridSpec(
    nrows=5, ncols=1, figure=fig,
    height_ratios=[4, 2.5, 2.5, 2.5, 2.5],
    hspace=0.12,
)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2], sharex=ax1)
ax3 = fig.add_subplot(gs[3], sharex=ax1)
ax4 = fig.add_subplot(gs[4], sharex=ax1)

for ax in [ax1, ax2, ax3]:
    plt.setp(ax.get_xticklabels(), visible=False)
    ax.set_xlabel("")

def fmt_time_axis(ax):
    ax.set_xlim(START, END)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
    ax.grid(True, which="minor", linestyle=":",  linewidth=0.3, alpha=0.3)


# ── (a) Double Mass Curve ────────────────────────────────────────────────────
dm_max = max(df["Cum_Obs_m3"].max(), df["Cum_Sim_m3"].max())
ax0.plot([0, dm_max], [0, dm_max], "k--", alpha=0.6, lw=1.0, label="1:1 line")
ax0.plot(df["Cum_Obs_m3"], df["Cum_Sim_m3"],
         color="blue", lw=1.5, label="Cumulative Sim vs Obs")
ax0.set_xlabel("Cumulative Observed Volume (m³)")
ax0.set_ylabel("Cumulative Simulated Volume (m³)")
ax0.set_title("(a) Double Mass Curve", loc="left", fontweight="bold")
ax0.set_xlim(0, dm_max)
ax0.set_ylim(0, dm_max)
ax0.legend(loc="lower right")
ax0.grid(True, alpha=0.5)
ax0.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
ax0.text(
    0.04, 0.93,
    f"Slope = {slope:.3f}\nPBIAS = {pbias:.2f} %\nR²    = {r2:.3f}",
    transform=ax0.transAxes, fontsize=7.5, va="top",
    bbox=dict(facecolor="white", alpha=0.85, boxstyle="round,pad=0.3")
)


# ── (b) Daily Flow Deviation ─────────────────────────────────────────────────
colors_flow = np.where(df["Flow_Dev_m3s"] >= 0, "#1f77b4", "#d62728")
ax1.bar(df["Zeit"], df["Flow_Dev_m3s"],
        color=colors_flow, width=np.timedelta64(1, "D"), align="center")
ax1.axhline(0, color="black", lw=0.8)
ax1.set_ylabel("Dev. (m³/s)")
ax1.set_title("(b) Daily Flow Deviation: Sim − Obs",
              loc="left", fontweight="bold")
ax1.text(0.01, 0.97,
         "Blue = overestimation  |  Red = underestimation",
         transform=ax1.transAxes, fontsize=6.5, va="top",
         bbox=dict(facecolor="white", alpha=0.7, boxstyle="round,pad=0.2"))
fmt_time_axis(ax1)


# ── (c) Cumulative Precipitation ─────────────────────────────────────────────
ax2.plot(df_precip_daily["datetime"], df_precip_daily["precip_cum"],
         color="darkblue", lw=1.4)
ax2.set_ylabel("Cum. Precip (mm)")
ax2.set_title("(c) Cumulative Precipitation", loc="left", fontweight="bold")
fmt_time_axis(ax2)


# ── (d) Reservoir Water Level  [visual only] ─────────────────────────────────
mask    = ~df_res_daily["WL"].isna()
dates_v = df_res_daily["Zeit"][mask]
wl_v    = df_res_daily["WL"][mask]

ax3.plot(dates_v, wl_v, color="red", lw=1.0, zorder=1)
colors_wl = np.where(wl_v.values >= VOLLSTAU, "green", "blue")
ax3.scatter(dates_v, wl_v, color=colors_wl, s=6, edgecolor="none", zorder=2)
ax3.axhline(VOLLSTAU, color="black", lw=0.8, linestyle="--", alpha=0.6,
            label=f"Full supply ({VOLLSTAU} m)")
ax3.set_ylabel("WL (m)")
ax3.set_title("(d) Neuer Teich Water Level", loc="left", fontweight="bold")
wl_min, wl_max = wl_v.min(), wl_v.max()
ax3.set_yticks(np.arange(np.floor(wl_min), np.ceil(wl_max) + 1, 1.0))
ax3.legend(loc="lower right", fontsize=6.5)
fmt_time_axis(ax3)


# ── (e) Artificial Reservoir Storage ─────────────────────────────────────────
# Clean: storage line + fill only, no twin axis, no flux bars
ax4.fill_between(df["Zeit"], df["ARR_Storage_m3"],
                 alpha=0.20, color="teal")
ax4.plot(df["Zeit"], df["ARR_Storage_m3"],
         color="teal", lw=1.5, label="Storage (m³)", zorder=3)

# Peak annotation
ax4.annotate(
    f"Peak: {peak_storage:,.0f} m³\n({peak_date.strftime('%b %Y')})",
    xy=(peak_date, peak_storage),
    xytext=(0, 16), textcoords="offset points",
    fontsize=6.5, ha="center", color="teal",
    arrowprops=dict(arrowstyle="->", color="teal", lw=0.8),
)

# Known catchment capacity line (optional)
if CATCHMENT_RESERVOIR_CAPACITY_M3 is not None:
    ax4.axhline(CATCHMENT_RESERVOIR_CAPACITY_M3,
                color="black", lw=1.0, linestyle=":",
                label=f"Catchment capacity "
                      f"({CATCHMENT_RESERVOIR_CAPACITY_M3:,.0f} m³)")

ax4.set_ylabel("Storage (m³)")
ax4.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
)
ax4.set_xlabel("Year")
ax4.set_title(
    "(e) Artificial Reservoir Storage  [Peak = min. capacity needed]",
    loc="left", fontweight="bold"
)
ax4.legend(loc="upper left", fontsize=6.5)
fmt_time_axis(ax4)


# ── Save ─────────────────────────────────────────────────────────────────────
fig.savefig("double_mass_figure.png", bbox_inches="tight", dpi=300)
print("Figure saved → double_mass_figure.png")
plt.show()